# Plant Disease Classifier - Model Training
Train a Random Forest classifier to detect plant leaf diseases using the PlantVillage dataset.

## 1. Install and Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib
import os

## 2. Upload Dataset
Upload PlantVillage dataset to Google Drive and mount it, or upload directly to Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this path to where your dataset is in Google Drive
DATASET_DIR = '/content/drive/MyDrive/dataset'
MAX_IMAGES = 15
IMG_SIZE = 64

## 3. Define Feature Extraction
Extract HSV color histograms, gradient features, and color ratios from each leaf image.

In [ ]:
def extract_features(pil_img):
    # Resize and convert to RGB
    img = pil_img.resize((IMG_SIZE, IMG_SIZE)).convert('RGB')
    arr = np.array(img, dtype=np.float32)
    r, g, b = arr[:,:,0], arr[:,:,1], arr[:,:,2]

    # Convert RGB to HSV manually
    arr_norm = arr / 255.0
    r_n, g_n, b_n = arr_norm[:,:,0], arr_norm[:,:,1], arr_norm[:,:,2]
    cmax = np.max(arr_norm, axis=2)
    cmin = np.min(arr_norm, axis=2)
    delta = cmax - cmin + 1e-7

    h = np.zeros_like(cmax)
    mask_r = cmax == r_n
    mask_g = (cmax == g_n) & ~mask_r
    mask_b = ~mask_r & ~mask_g
    h[mask_r] = 60 * (((g_n[mask_r] - b_n[mask_r]) / delta[mask_r]) % 6)
    h[mask_g] = 60 * (((b_n[mask_g] - r_n[mask_g]) / delta[mask_g]) + 2)
    h[mask_b] = 60 * (((r_n[mask_b] - g_n[mask_b]) / delta[mask_b]) + 4)

    s = np.where(cmax == 0, 0, delta / (cmax + 1e-7))
    v = cmax

    h_scaled = (h / 360 * 255).astype(np.float32)
    s_scaled = (s * 255).astype(np.float32)
    v_scaled = (v * 255).astype(np.float32)

    # Grayscale
    gray = 0.299 * r + 0.587 * g + 0.114 * b

    features = []

    # HSV color histograms (32 bins each = 96 features)
    for ch in [h_scaled, s_scaled, v_scaled]:
        hist, _ = np.histogram(ch, bins=32, range=(0, 256))
        hist = hist.astype(np.float32) / (hist.sum() + 1e-7)
        features.extend(hist.tolist())

    # Mean and std of each channel (8 features)
    for ch in [h_scaled, s_scaled, v_scaled, gray]:
        features.append(float(ch.mean()))
        features.append(float(ch.std()))

    # Sobel edge gradients (2 features)
    gy = np.zeros_like(gray)
    gx = np.zeros_like(gray)
    gy[1:-1,:] = gray[2:,:] - gray[:-2,:]
    gx[:,1:-1] = gray[:,2:] - gray[:,:-2]
    mag = np.sqrt(gx**2 + gy**2)
    features.append(float(mag.mean()))
    features.append(float(mag.std()))

    # Color ratios (3 features)
    total = r + g + b + 1e-7
    features.append(float((g / total).mean()))
    features.append(float((r / total).mean()))
    features.append(float((b / total).mean()))

    return np.array(features, dtype=np.float32)

print(f'Feature vector size: 109 (96 histogram + 8 stats + 2 gradient + 3 color ratio)')

## 4. Load Dataset and Extract Features

In [ ]:
X, y = [], []

classes = sorted([d for d in os.listdir(DATASET_DIR)
                  if os.path.isdir(os.path.join(DATASET_DIR, d))
                  and not d.startswith('.')])

print(f'Found {len(classes)} classes')

for cls_name in classes:
    cls_dir = os.path.join(DATASET_DIR, cls_name)
    images = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    images = images[:MAX_IMAGES]

    count = 0
    for img_name in images:
        path = os.path.join(cls_dir, img_name)
        try:
            img = Image.open(path)
            feat = extract_features(img)
            X.append(feat)
            y.append(cls_name)
            count += 1
        except:
            continue

    print(f'  [{count:>4} images] {cls_name}')

X = np.array(X)
y = np.array(y)
print(f'\nTotal: {len(X)} samples, {len(classes)} classes')
print(f'Feature vector size: {X.shape[1]}')

## 5. Dataset Distribution

In [ ]:
df = pd.DataFrame({'class': y, 'plant': [c.split('___')[0] for c in y]})
plant_counts = df['plant'].value_counts()

plt.figure(figsize=(12, 5))
plt.bar(range(len(plant_counts)), plant_counts.values, color='#40916c')
plt.xticks(range(len(plant_counts)), plant_counts.index, rotation=45, ha='right', fontsize=8)
plt.ylabel('Number of Images')
plt.title('Dataset Distribution by Plant')
plt.tight_layout()
plt.show()

print(df['plant'].value_counts())

## 6. Split Train/Test Data

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'Train: {len(X_train)} samples')
print(f'Test:  {len(X_test)} samples')

## 7. Train Random Forest Model

In [ ]:
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    n_jobs=-1,
    random_state=42
)

clf.fit(X_train, y_train)
print('Training complete!')

## 8. Evaluate Model

In [ ]:
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Test Accuracy: {acc:.1%}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

## 9. Accuracy per Plant

In [ ]:
plants = sorted(set(c.split('___')[0] for c in le.classes_))
plant_acc = {}

for plant in plants:
    mask = np.array([le.classes_[i].startswith(plant) for i in y_test])
    if mask.sum() > 0:
        plant_acc[plant] = accuracy_score(y_test[mask], y_pred[mask])

names = list(plant_acc.keys())
accs = list(plant_acc.values())
colors = ['#2d6a4f' if a >= 0.5 else '#d62828' for a in accs]

plt.figure(figsize=(12, 5))
bars = plt.bar(range(len(names)), [a * 100 for a in accs], color=colors)
plt.xticks(range(len(names)), names, rotation=45, ha='right', fontsize=8)
plt.ylabel('Accuracy (%)')
plt.title('Model Accuracy per Plant')
plt.ylim(0, 100)
for bar, a in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f'{a:.0%}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.show()

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14, 12))
plt.imshow(cm, cmap='Greens', interpolation='nearest')
plt.title(f'Confusion Matrix ({len(le.classes_)} classes)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.colorbar(shrink=0.8)
plt.tight_layout()
plt.show()

## 11. Save Model

In [ ]:
model_data = {
    'classifier': clf,
    'label_encoder': le,
    'classes': classes,
    'feature_size': X.shape[1],
    'accuracy': acc
}

joblib.dump(model_data, 'plant_model.pkl', compress=3)
size_mb = os.path.getsize('plant_model.pkl') / (1024 * 1024)

print(f'Model saved: plant_model.pkl ({size_mb:.1f} MB)')
print(f'Accuracy: {acc:.1%} on {len(classes)} classes')

## 12. Summary

In [ ]:
print(f'''
========================================
  Plant Disease Classifier Results
========================================

  Total Classes:    {len(classes)}
  Total Samples:    {len(X)}
  Test Accuracy:    {acc:.1%}

  Model:            Random Forest
  Estimators:       100
  Max Depth:        20
  Features:         HSV + Gradient + Color
  Feature Size:     109
========================================
''')